In [0]:
from pyspark.sql.functions import col, sum, countDistinct, avg, lag
from pyspark.sql.window import Window

def limpiar_ventas(df):
    return df.filter(
        col("monto").isNotNull() & (col("monto") > 0)
    )

def enriquecer_ventas(ventas, productos):
    return ventas.join(productos, "producto_id", "left")

def calcular_resumen(df):
    resumen = df.groupBy("fecha").agg(
        sum("monto").alias("ventas_totales"),
        countDistinct("cliente_id").alias("clientes_unicos")
    )
    
    w = Window.orderBy("fecha")
    
    resumen = resumen.withColumn(
        "ventas_7d",
        avg("ventas_totales").over(w.rowsBetween(-6, 0))
    )
    
    resumen = resumen.withColumn(
        "ventas_prev_dia",
        lag("ventas_totales").over(w)
    )
    
    return resumen